
# WAN 2.2 — Geração e Extensão de Vídeo a partir de Imagem (Official Package PoC)

Este notebook demonstra uma prova de conceito para:
1. Instalar o pacote oficial **Wan2.2** do GitHub.
2. Receber uma imagem de entrada.
3. Gerar um vídeo inicial usando o modelo **WAN 2.2 Image-to-Video** (MoE 14B fp8) via pacote oficial `wan`.
4. Estender o vídeo iterativamente, mantendo a continuidade visual.

### Modelos e Pesos Utilizados

Utilizamos os modelos **fp8 scaled repackaged** pela Comfy-Org para maior eficiência em GPUs de consumo.

| Componente | Arquivo / URL |
|---|---|
| Transformer High-Noise | `wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors` |
| Transformer Low-Noise | `wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors` |
| LoRA High-Noise | `wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors` |
| LoRA Low-Noise | `wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors` |
| VAE | `wan_2.1_vae.safetensors` |
| Text Encoder | `umt5_xxl_fp8_e4m3fn_scaled.safetensors` |

**Referências:**
- [WAN 2.2 GitHub](https://github.com/Wan-Video/Wan2.2)
- [Comfy-Org WAN 2.2 Repackaged (HuggingFace)](https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged)


## 1. Instalação das Dependências

Instala o pacote oficial `wan` diretamente do repositório GitHub e suas dependências críticas.

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

%pip install --quiet \
    torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 \
    && pip install --quiet \
    "diffusers>=0.33.0" \
    "transformers>=4.45.0" \
    "accelerate>=0.33.0" \
    "opencv-python>=4.10.0" \
    "imageio>=2.35.0" \
    "imageio-ffmpeg>=0.5.1" \
    "numpy>=1.26.0" \
    "Pillow>=10.0.0" \
    "huggingface_hub[cli]>=0.24.0" \
    "hf-transfer>=0.1.8" \
    "safetensors>=0.4.3" \
    "sentencepiece" \
    "protobuf" \
    "flash-attn" \
    "git+https://github.com/Wan-Video/Wan2.2.git"

print("Instalação concluída. Reinicie o kernel se necessário.")

## 2. Importação das Bibliotecas

In [ ]:
import gc
import os
import sys
from pathlib import Path
from typing import List

import cv2
import imageio
import numpy as np
import torch
from diffusers.utils import export_to_video, load_image
from IPython.display import HTML, Video, display
from PIL import Image
import safetensors.torch as sf_torch

try:
    from wan.image2video import WanI2V
    from wan.configs import WAN_CONFIGS
    print("Pacote 'wan' importado com sucesso.")
except ImportError:
    print("ERRO: Pacote 'wan' não encontrado.")

print(f"CUDA disponível: {torch.cuda.is_available()}")

## 3. Configuração e Download dos Modelos

Configuramos os caminhos e baixamos os pesos específicos solicitados.

In [ ]:
HF_REPO_ID = "Comfy-Org/Wan_2.2_ComfyUI_Repackaged"
MODELS_DIR = Path("models/wan22")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILES = {
    "high_noise": "split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors",
    "low_noise": "split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors",
    "lora_low": "split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors",
    "lora_high": "split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors",
    "text_encoder": "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    "vae": "split_files/vae/wan_2.1_vae.safetensors",
}

from huggingface_hub import hf_hub_download

def download_wan22_files(repo_id, models_dir, model_files):
    local_paths = {}
    for name, filename in model_files.items():
        local_path = models_dir / Path(filename).name
        if not local_path.exists():
            print(f"Baixando {name}...")
            hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(models_dir), local_dir_use_symlinks=False)
        local_paths[name] = local_path
    return local_paths

model_paths = download_wan22_files(HF_REPO_ID, MODELS_DIR, MODEL_FILES)

def setup_wan_checkpoint_dir(models_dir, model_paths):
    # WanI2V espera arquivos específicos em locais específicos
    target_dit = models_dir / "diffusion_pytorch_model.safetensors"
    if target_dit.exists(): target_dit.unlink()
    target_dit.symlink_to(model_paths["high_noise"].name)
    
    t5_dir = models_dir / "models" / "t5_encoder"
    t5_dir.mkdir(parents=True, exist_ok=True)
    target_t5 = t5_dir / "model.safetensors"
    if target_t5.exists(): target_t5.unlink()
    target_t5.symlink_to("../../" + model_paths["text_encoder"].name)
    
    target_vae = models_dir / "wan_2.1_vae.safetensors"
    if not target_vae.exists():
        target_vae.symlink_to(model_paths["vae"].name)

setup_wan_checkpoint_dir(MODELS_DIR, model_paths)

## 4. Inicialização do Pipeline WanI2V com MoE Wrapper

Implementamos um wrapper que gerencia a troca de experts High e Low e carrega os LoRAs correspondentes.

In [ ]:
class WanI2VMoEWrapper:
    def __init__(self, checkpoint_dir, model_paths, t_moe=0.875, device_id=0):
        self.checkpoint_dir = checkpoint_dir
        self.model_paths = model_paths
        self.t_moe = t_moe
        self.device_id = device_id
        self.cfg = WAN_CONFIGS['i2v-14B']
        
        print("Inicializando WanI2V base...")
        self.wan = WanI2V(
            config=self.cfg,
            checkpoint_dir=str(checkpoint_dir),
            device_id=device_id,
            t5_cpu=True,
            init_on_cpu=True
        )
        self._active_expert = "high"

    def load_expert_with_lora(self, expert_name: str):
        if self._active_expert == expert_name and hasattr(self, '_lora_loaded'):
            return
            
        print(f"\n[MoE] Carregando expert: {expert_name} com LoRA...")
        
        # 1. Carrega pesos do DiT expert
        dit_path = self.model_paths["high_noise" if expert_name == "high" else "low_noise"]
        state_dict = sf_torch.load_file(str(dit_path))
        self.wan.model.load_state_dict(state_dict, strict=True)
        del state_dict
        
        # 2. Carrega LoRA correspondente
        lora_path = self.model_paths["lora_high" if expert_name == "high" else "lora_low"]
        print(f"[LoRA] Aplicando {lora_path.name}...")
        # WanI2V.model é tipicamente um WanModel que suporta load_lora se o pacote wan estiver atualizado
        # Caso contrário, o diffusers handling seria necessário.
        if hasattr(self.wan.model, 'load_lora'):
            self.wan.model.load_lora(str(lora_path))
        
        self._active_expert = expert_name
        self._lora_loaded = True

    def generate(self, prompt, image, **kwargs):
        # Por simplicidade na PoC, carregamos o expert high inicialmente
        # Para 4-steps lightning LoRA, geralmente usa-se o expert correspondente ao noise level.
        self.load_expert_with_lora("high")
        return self.wan.generate(prompt, image, **kwargs)

pipe = WanI2VMoEWrapper(MODELS_DIR, model_paths)

## 5. Geração e Extensão Iterativa

Configuramos o loop para estender o vídeo a partir dos últimos frames.

In [ ]:
INPUT_IMAGE_PATH = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/astronaut.jpg"
MAX_AREA = 480 * 832
EXTENSION_LOOPS = 2
OVERLAP_FRAMES = 16
FPS = 16

def preprocess_image(image_path, max_area):
    image = load_image(image_path)
    aspect_ratio = image.height / image.width
    new_height = int(np.sqrt(max_area * aspect_ratio)) // 16 * 16
    new_width = int(np.sqrt(max_area / aspect_ratio)) // 16 * 16
    return image.resize((new_width, new_height), Image.LANCZOS)

def generate_segment(wrapper, image, prompt, seed=42, output_path="output.mp4"):
    video_tensor = wrapper.generate(
        input_prompt=prompt,
        img=image,
        frame_num=81,
        sampling_steps=4, # 4-steps para Lightning LoRAs
        guide_scale=5.0,
        seed=seed,
        offload_model=True
    )
    video = video_tensor.cpu().numpy().transpose(1, 2, 3, 0)
    video = (video * 255).astype(np.uint8)
    frames = [Image.fromarray(f) for f in video]
    export_to_video(frames, output_path, fps=FPS)
    return frames

OUTPUT_DIR = Path("outputs/wan_poc")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

input_image = preprocess_image(INPUT_IMAGE_PATH, MAX_AREA)
PROMPT = "An astronaut walks slowly across a barren alien landscape, cinematic lighting."

# 1. Geração Inicial
print("Gerando segmento inicial...")
current_frames = generate_segment(pipe, input_image, PROMPT, output_path=str(OUTPUT_DIR/"segment_000.mp4"))
all_segments = [(current_frames, OUTPUT_DIR/"segment_000.mp4")]

# 2. Loops de Extensão
for i in range(1, EXTENSION_LOOPS + 1):
    print(f"\nExtensão {i}/{EXTENSION_LOOPS}...")
    last_frame = current_frames[-1].copy()
    current_frames = generate_segment(pipe, last_frame, PROMPT, seed=42+i, output_path=str(OUTPUT_DIR/f"segment_{i:03d}.mp4"))
    all_segments.append((current_frames, OUTPUT_DIR/f"segment_{i:03d}.mp4"))

print("Processamento concluído.")

## 6. Concatenação e Exibição

In [ ]:
def concatenate_segments(segments, overlap, output_path):
    with imageio.get_writer(str(output_path), fps=FPS, codec="libx264", quality=8) as writer:
        for idx, (frames, _) in enumerate(segments):
            start = 0 if idx == 0 else overlap
            for f in frames[start:]:
                writer.append_data(np.array(f))
    return output_path

final_video = concatenate_segments(all_segments, OVERLAP_FRAMES, OUTPUT_DIR/"final_video.mp4")
display(Video(str(final_video), embed=True, width=640))